#Init

In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

#Read from Bronze Table

In [0]:
df = spark.table('workspace.bronze.erp_cust_az12')

#Data Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, T.StringType):
        trimmed_col = F.trim(F.col(field.name))
        
        df = df.withColumn(
            field.name, 
            F.when(trimmed_col == "", F.lit(None).cast("string"))
             .otherwise(trimmed_col)
        )

##Customer ID Cleanup

In [0]:
df = df.withColumn(
    "cid",
    F.when(F.col("cid").startswith("NAS"),
           F.substring(F.col("cid"), 4, F.length(F.col("cid"))))
     .otherwise(F.col("cid"))
)

##Birthdate Validation

In [0]:
df = df.withColumn(
    "bdate",
    F.when(F.col("bdate") > F.current_date(), None)
     .otherwise(F.col("bdate"))
)

##Gender Normalization

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(F.col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(F.col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)

## Renaming columns

In [0]:
RENAME_MAP = {
    "CID": "customer_id",
    "BDATE": "birth_date",
    "GEN": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity checks Dataframe

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("workspace.silver.erp_customer")

##Sanity checks Silver Table

In [0]:
%sql
SELECT * FROM silver.erp_customer limit 10